# PSM Experiment — Kaggle Notebook

**Predictive Semantic Memory (PSM)** experiment runner for Kaggle.

This notebook:
1. Loads a Kaggle-hosted LLM (no external API key needed)
2. Runs a **smoke test** (synthetic questions) to validate the pipeline end-to-end
3. Runs the **full TriviaQA experiment** (smoke + pilot profiles)
4. Saves **all results as human-readable CSV files** ready to push to GitHub

---
### ⚡ Quick-start
1. Open **Notebook settings → Models** and add your preferred model  
   *(Qwen2 7B Instruct GPTQ-INT4 is a great choice — accept the Kaggle licence once)*
2. Set `KAGGLE_MODEL_PATH` in the **Config** cell below to match the path Kaggle assigns
3. Run all cells top-to-bottom (**Run → Run All**)


## 1 · Install required packages

Kaggle pre-installs `torch` and `transformers`. The cells below install the few extra packages the PSM pipeline needs.

In [ ]:
%%capture install_output
import subprocess, sys

PACKAGES = [
    "faiss-cpu",
    "sentence-transformers",
    "rank-bm25",
    "datasets",
    "rouge-score",
    "nltk",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])
print("✓ All packages installed.")


In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("✓ NLTK data ready.")


## 2 · Configuration

Edit the variables below to match **your** Kaggle notebook setup.

### Common Kaggle model paths (after adding the model in notebook settings)
| Model | Kaggle path |
|-------|-------------|
| **Qwen2 7B Instruct GPTQ-INT4** *(recommended)* | `/kaggle/input/qwen-lm/qwen2/transformers/7b-instruct-gptq-int4/1` |
| Gemma 7B IT | `/kaggle/input/gemma/transformers/7b-it/3` |
| Phi-2 | `/kaggle/input/phi-2/transformers/default/1` |
| Mistral 7B Instruct v0.2 | `/kaggle/input/mistral-7b-instruct-v0.2/transformers/default/1` |
| Llama 3 8B Instruct | `/kaggle/input/llama-3/transformers/8b-instruct/1` |

> **Tip:** After adding a model, click the ⓘ icon next to its name to see the exact path.


In [ ]:
import os
from pathlib import Path

# ── CHANGE THIS to your model path ──────────────────────────────────────────
KAGGLE_MODEL_PATH = "/kaggle/input/qwen-lm/qwen2/transformers/7b-instruct-gptq-int4/1"

# Validate model path exists before proceeding
if not Path(KAGGLE_MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Model path not found: {KAGGLE_MODEL_PATH}\n"
        "To fix:\n"
        "  1. Open Notebook settings → Models → add your preferred model\n"
        "  2. Click the ⓘ icon next to the model to see its exact path\n"
        "  3. Update KAGGLE_MODEL_PATH above and re-run this cell\n"
        "\n"
        "Common paths:\n"
        "  Qwen2 7B Instruct GPTQ-INT4 : /kaggle/input/qwen-lm/qwen2/transformers/7b-instruct-gptq-int4/1\n"
        "  Gemma 7B IT : /kaggle/input/gemma/transformers/7b-it/3\n"
        "  Phi-2       : /kaggle/input/phi-2/transformers/default/1\n"
        "  Mistral 7B  : /kaggle/input/mistral-7b-instruct-v0.2/transformers/default/1"
    )

# ── Experiment settings ──────────────────────────────────────────────────────
SMOKE_SIZE      = 10    # synthetic questions for the smoke test
CONFIDENCE_THRESHOLD = 0.55

# ── Output directory (all CSVs land here) ────────────────────────────────────
OUTPUT_BASE = "/kaggle/working/psm_results"

# Apply environment variables so the PSM pipeline picks them up
os.environ["PSM_LLM_BACKEND"]       = "kaggle"
os.environ["PSM_KAGGLE_MODEL_PATH"] = KAGGLE_MODEL_PATH
os.environ["PSM_NUM_PREDICT"]       = "64"    # max new tokens per LLM call
os.environ["PSM_TEMPERATURE"]       = "0.1"   # near-greedy for factual Q&A

print(f"LLM backend  : {os.environ['PSM_LLM_BACKEND']}")
print(f"Model path   : {os.environ['PSM_KAGGLE_MODEL_PATH']}")
print(f"Output base  : {OUTPUT_BASE}")
print("✓ Model path verified and configuration applied.")


## 3 · Import PSM modules

In [ ]:
import sys
from pathlib import Path

# Point Python at the PSM source tree
PSM_ROOT = None
candidates = [
    Path("/kaggle/input/datasets/kausikvaibhavpatra/psm-experiment/PSM_Experiment_Setup_Research"),
    Path("/kaggle/input/psm-experiment-research-setup-updated/PSM_Experiment_Setup_Research"),
    Path("/kaggle/input/psm-experiment/PSM_Experiment_Setup_Research"),
    Path.cwd() / "PSM_Experiment_Setup_Research",
    Path.cwd(),
]
for candidate in candidates:
    if (candidate / "router").exists() and (candidate / "llm").exists():
        PSM_ROOT = candidate
        break

if PSM_ROOT is None:
    raise RuntimeError(
        "PSM source tree not found. Ensure one of the following:\n"
        "  • The PSM dataset is attached to this notebook (Add-ons → Datasets)\n"
        "  • Or the notebook is inside the cloned PSM repo directory\n"
        "Expected to find a 'router/' and 'llm/' folder inside PSM_Experiment_Setup_Research."
    )

sys.path.insert(0, str(PSM_ROOT))
print(f"PSM root: {PSM_ROOT}")

# Verify core imports
from router.router import Router                          # noqa: E402
from evaluation.metrics import exact_match, token_f1, rouge_l, bleu, contains_any_answer
from run_psm_experiments import PSMExperimentRunner, ExperimentProfile
from triviaqa_smoke_run_fast import PSMSmokeRunFast

print("✓ All PSM modules imported successfully.")


## 4 · Smoke test (synthetic questions)

This cell runs **10 synthetic questions** through the full pipeline:
*embedding → FAISS retrieval → LLM generation → metrics*.

It must pass before the full TriviaQA experiment runs.  
✅ Pass = all 10 questions complete without errors.


In [ ]:
import os, shutil
from pathlib import Path
import pandas as pd

SMOKE_OUT = Path(OUTPUT_BASE) / "smoke_test"
shutil.rmtree(SMOKE_OUT, ignore_errors=True)   # fresh run each time

# PSMSmokeRunFast writes its outputs to smoke_results/<run_id>/<timestamp>/
# We change cwd temporarily so relative paths land inside our output folder
original_cwd = os.getcwd()
os.makedirs(SMOKE_OUT, exist_ok=True)
os.chdir(SMOKE_OUT)

try:
    smoke = PSMSmokeRunFast(smoke_size=SMOKE_SIZE, run_id="kaggle_smoke")
    smoke_metrics = smoke.run()
finally:
    os.chdir(original_cwd)

if smoke_metrics is None:
    raise RuntimeError("Smoke test produced no metrics — check errors above.")

print()
print("=" * 60)
print("SMOKE TEST RESULT")
print("=" * 60)
print(f"  Total questions : {smoke_metrics['total_queries']}")
print(f"  Avg EM          : {smoke_metrics['avg_em']:.3f}")
print(f"  Avg F1          : {smoke_metrics['avg_f1']:.3f}")
print(f"  Avg Confidence  : {smoke_metrics['avg_confidence']:.3f}")
print(f"  Memory rate     : {smoke_metrics['memory_path_rate']:.1%}")
print(f"  Retrieval rate  : {smoke_metrics['retrieval_path_rate']:.1%}")
print("=" * 60)
print("✓ SMOKE TEST PASSED — pipeline is healthy, proceeding to full experiment.")


### 4a · Smoke test results (CSV preview)

In [ ]:
import glob, pandas as pd

smoke_pred_files = glob.glob(str(SMOKE_OUT / "**" / "predictions.jsonl"), recursive=True)

if smoke_pred_files:
    smoke_df = pd.read_json(smoke_pred_files[0], lines=True)
    print(f"Predictions shape: {smoke_df.shape}")
    display(smoke_df[["query_id", "question", "generated_answer", "correct_answer",
                       "em", "f1", "confidence", "source"]].head(SMOKE_SIZE))
else:
    print("No predictions file found (check smoke test cell for errors).")


## 5 · Full experiment — TriviaQA

Runs two profiles back-to-back:

| Profile | Questions | Corpus pool | Description |
|---------|-----------|-------------|-------------|
| `smoke` | 5 | 140 docs | Fast sanity check on real data |
| `pilot` | 8 | 260 docs | Small but realistic evaluation |

All results are written to `OUTPUT_BASE/experiment_runs/` as:
- `predictions.csv` — per-question results with all metrics
- `summary_metrics.csv` — flat one-row summary (easy to open in Excel)
- `summary_metrics.json` — full nested summary
- `all_runs_metrics.csv` — aggregated table across all profiles


In [ ]:
import os, shutil
from pathlib import Path

EXP_OUT = Path(OUTPUT_BASE) / "experiment_runs"
shutil.rmtree(EXP_OUT, ignore_errors=True)
EXP_OUT.mkdir(parents=True, exist_ok=True)

original_cwd = os.getcwd()
os.chdir(Path(OUTPUT_BASE))   # runner uses relative paths internally

try:
    runner = PSMExperimentRunner(
        base_results_dir=str(EXP_OUT),
        threshold=CONFIDENCE_THRESHOLD,
    )
    experiment_results = runner.execute()
finally:
    os.chdir(original_cwd)

print()
print("=" * 60)
print("EXPERIMENT COMPLETE")
print(f"  Runs completed : {len(experiment_results['runs'])}")
print(f"  Output dir     : {EXP_OUT}")
print("=" * 60)


## 6 · Evaluation metrics — all CSV results

In [ ]:
import glob, pandas as pd
from pathlib import Path

EXP_OUT = Path(OUTPUT_BASE) / "experiment_runs"

# ── Per-run predictions ──────────────────────────────────────────────────────
pred_csvs = sorted(EXP_OUT.glob("**/predictions.csv"))
print(f"Found {len(pred_csvs)} predictions.csv file(s)")
print()

all_preds = []
for p in pred_csvs:
    run_name = p.parent.name
    df = pd.read_csv(p)
    df["run_name"] = run_name
    all_preds.append(df)
    print(f"─── {run_name}  ({len(df)} rows) ───")
    print(df[["query_id", "question", "generated_answer", "correct_answer",
              "em", "f1", "rouge_l", "bleu", "confidence", "source",
              "latency_s"]].to_string(index=False, max_colwidth=55))
    print()

if all_preds:
    combined_preds = pd.concat(all_preds, ignore_index=True)
    combined_path = EXP_OUT / "all_predictions_combined.csv"
    combined_preds.to_csv(combined_path, index=False)
    print(f"✓ Combined predictions saved → {combined_path}")


In [ ]:
# ── Summary metrics (flat CSV, one row per run) ──────────────────────────────
summary_csvs = sorted(EXP_OUT.glob("**/summary_metrics.csv"))
print(f"Found {len(summary_csvs)} summary_metrics.csv file(s)")
print()

all_summaries = []
for s in summary_csvs:
    df = pd.read_csv(s)
    all_summaries.append(df)

if all_summaries:
    summary_df = pd.concat(all_summaries, ignore_index=True)
    cols_to_show = [
        "run_id", "num_questions", "avg_em", "avg_f1", "avg_rouge_l", "avg_bleu",
        "avg_confidence", "avg_latency_s", "hallucination_rate",
        "memory_path_rate", "retrieval_path_rate", "final_memory_size",
    ]
    print(summary_df[cols_to_show].to_string(index=False))
    print()

# ── Master aggregate CSV ─────────────────────────────────────────────────────
master_csv = EXP_OUT / "all_runs_metrics.csv"
if master_csv.exists():
    master_df = pd.read_csv(master_csv)
    print("─── all_runs_metrics.csv (aggregate) ───")
    print(master_df.to_string(index=False))


## 7 · Output files — ready to push to GitHub

In [ ]:
import os
from pathlib import Path

OUTPUT_BASE_PATH = Path(OUTPUT_BASE)

print("=" * 70)
print("ALL OUTPUT FILES")
print("=" * 70)

csv_files = sorted(OUTPUT_BASE_PATH.rglob("*.csv"))
json_files = sorted(OUTPUT_BASE_PATH.rglob("*.json"))
txt_files  = sorted(OUTPUT_BASE_PATH.rglob("*.txt"))

print(f"\n📊  CSV files ({len(csv_files)}) — human-readable, GitHub-friendly:")
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f"    {f.relative_to(OUTPUT_BASE_PATH)}  ({size_kb:.1f} KB)")

print(f"\n📋  JSON files ({len(json_files)}):")
for f in json_files:
    size_kb = f.stat().st_size / 1024
    print(f"    {f.relative_to(OUTPUT_BASE_PATH)}  ({size_kb:.1f} KB)")

print(f"\n📝  Text reports ({len(txt_files)}):")
for f in txt_files:
    size_kb = f.stat().st_size / 1024
    print(f"    {f.relative_to(OUTPUT_BASE_PATH)}  ({size_kb:.1f} KB)")

print()
print("=" * 70)
print("To push to GitHub:")
print("  1. Download the output folder from Kaggle (⬇ Output tab)")
print("  2. Copy the CSVs into your repo under results/<run_id>/")
print("  3. git add results/ && git commit -m 'Add experiment results' && git push")
print("=" * 70)


## 8 · (Optional) Larger experiment

Uncomment and adjust the cell below to run a larger evaluation.  
Increase `num_questions` and `corpus_pool_size` to match your Kaggle session GPU quota.


In [ ]:
# # Uncomment to run a larger experiment
#
# from run_psm_experiments import PSMExperimentRunner, ExperimentProfile
# import os
# from pathlib import Path
#
# LARGE_OUT = Path(OUTPUT_BASE) / "large_experiment"
# LARGE_OUT.mkdir(parents=True, exist_ok=True)
# original_cwd = os.getcwd()
# os.chdir(Path(OUTPUT_BASE))
#
# try:
#     runner = PSMExperimentRunner(
#         base_results_dir=str(LARGE_OUT),
#         threshold=CONFIDENCE_THRESHOLD,
#     )
#     # Override profiles for a larger run
#     profiles = [
#         ExperimentProfile(name="medium",  num_questions=25,  corpus_pool_size=500,  max_chunks=1500),
#         ExperimentProfile(name="full",    num_questions=100, corpus_pool_size=1000, max_chunks=3000),
#     ]
#     for p in profiles:
#         runner.run_profile(p)
# finally:
#     os.chdir(original_cwd)
